# Section IV Taxonomy Evidence Lab (Full-Scan + JSON/Markdown Fusion)

This notebook builds the evidence layer for Section IV (Taxonomy).
- Full scan: processed markdowns + O_ISAC JSON
- Variant-aware retrieval (lexical + fuzzy + LLM entailment)
- Two-model flow:
  - Pass-1 model (fast): broad classification over all hits
  - Pass-2 model (strict): only escalated uncertain hits
- Outputs: evidence CSVs, cluster maps, taxonomy data skeletons

Usage note:
- Tune model names and per-model RPM in `# @title 3. Config`.
- Keep `RESUME=True` for long runs; checkpoints are under `analysis/IV_ev_v1/checkpoints`.


In [43]:
# @title 1. Install Dependencies
!pip install -q groq rapidfuzz tqdm


In [44]:
# @title 2. Setup & Mount Drive
from google.colab import drive, userdata
import os, re, json, glob
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print('Working dir:', os.getcwd())
else:
    print('Path not found:', BASE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [45]:
# @title 3. Config
PROCESSED_MD_DIR = Path('data/proc_markdowns')
JSON_DIR = Path('data/ext_res_v4')
UNIFIED_JSON = JSON_DIR / 'extraction_v4_unified.json'
OUTPUT_DIR = Path('analysis/IV_ev_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = 'FULL_RESCAN'  # 1B selected
NORMALIZE_LABELS = True      # 3A selected

TARGET_PAPERS = None  # Full scan by default
LIMIT = None          # Full scan by default

# LLM execution controls (limit-aware)
LLM_CALLS = True

# Two-model strategy
MODEL_PASS1 = 'meta-llama/llama-4-scout-17b-16e-instruct'   # fast sweep
MODEL_PASS2 = 'llama-3.3-70b-versatile'                      # strict recheck
MODEL_VARIANT_GEN = MODEL_PASS1                              # variant generation model
USE_ESCALATION = True
ESCALATE_LABELS = {'INDIRECT', 'NONE', 'WEAK'}

# Per-model rate limits (requests/min), adjust to your Groq profile
RPM_BY_MODEL = {
    MODEL_PASS1: 120,
    MODEL_PASS2: 40,
    MODEL_VARIANT_GEN: 120,
}
DEFAULT_RPM = 30

MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

# Retrieval and batching controls
MAX_VARIANTS_PER_CONCEPT = 10
MAX_HITS_PER_CONCEPT_PER_PAPER = 5
MAX_CONTEXT_CHARS = 1200
CLASSIFY_CHUNK_SIZE = 4
BATCH_SIZE_PAPERS = 10

# Checkpoint controls
RESUME = True
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_MAP_PATH = Path('analysis/II_sch_map.md')
GOV_PATH = Path('analysis/II_met_gov.md')

schema_text = SCHEMA_MAP_PATH.read_text(encoding='utf-8', errors='ignore')
gov_text = GOV_PATH.read_text(encoding='utf-8', errors='ignore')

print('Config ready. Output:', OUTPUT_DIR)
print('Run profile:', RUN_PROFILE)
print('Pass-1 model:', MODEL_PASS1)
print('Pass-2 model:', MODEL_PASS2)


Config ready. Output: analysis/IV_ev_v2
Run profile: FULL_RESCAN
Pass-1 model: meta-llama/llama-4-scout-17b-16e-instruct
Pass-2 model: llama-3.3-70b-versatile


In [46]:
# @title 4. Load O_ISAC JSON Index
def load_json_index(json_dir: Path):
    index = {}
    for p in sorted(json_dir.glob('O_ISAC_*_v4.json')):
        paper_id = p.stem.replace('_v4','')
        try:
            index[paper_id] = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
        except Exception as e:
            index[paper_id] = {'_error': str(e)}
    unified = None
    if UNIFIED_JSON.exists():
        unified = json.loads(UNIFIED_JSON.read_text(encoding='utf-8', errors='ignore'))
    return index, unified

json_index, unified_json = load_json_index(JSON_DIR)
print('JSON files loaded:', len(json_index))
print('Unified JSON:', 'yes' if unified_json else 'no')


JSON files loaded: 221
Unified JSON: yes


In [47]:
# @title 5. Load Processed Markdowns (Canonical per paper)
def canonical_md_path(paths, paper_id):
    scored = []
    for p in paths:
        p = Path(p)
        score = 0
        if (p.parent / 'visual_analysis.txt').exists():
            score += 3
        if p.parent.name == paper_id and p.parent.parent.name == paper_id:
            score += 2
        score += len(p.parts) * 0.1
        scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None

def load_processed_markdowns(target_ids=None, limit=None):
    search_path = PROCESSED_MD_DIR
    all_files = list(search_path.rglob('*.md'))
    md_files = [p for p in all_files if 'O_ISAC_' in p.name]

    grouped = {}
    for p in md_files:
        m = re.search(r'(O_ISAC_\d+)', p.name)
        if not m:
            continue
        paper_id = m.group(1)
        if target_ids and paper_id not in target_ids:
            continue
        grouped.setdefault(paper_id, []).append(p)

    records = []
    for i, (paper_id, paths) in enumerate(sorted(grouped.items())):
        if limit and i >= limit:
            break
        canon = canonical_md_path(paths, paper_id)
        if not canon:
            continue
        text = canon.read_text(encoding='utf-8', errors='ignore')
        lines = text.splitlines()
        va_path = canon.parent / 'visual_analysis.txt'
        va_text = va_path.read_text(encoding='utf-8', errors='ignore') if va_path.exists() else ''
        records.append({
            'paper_id': paper_id,
            'md_path': str(canon),
            'text': text,
            'lines': lines,
            'visual_analysis': va_text
        })
    return records

papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)
print('Markdown papers loaded:', len(papers))


Markdown papers loaded: 221


In [48]:
# @title 6. Heading Map + Context Utils
def build_heading_map(lines):
    current = []
    heading_map = {}
    for i, line in enumerate(lines):
        if line.startswith('#'):
            level = len(line) - len(line.lstrip('#'))
            title = line.strip('#').strip()
            if level <= len(current):
                current = current[:level-1]
            current.append(title)
        heading_map[i] = ' > '.join(current) if current else 'no_heading'
    return heading_map

def get_context(lines, idx, window=2):
    start = max(0, idx - window)
    end = min(len(lines), idx + window + 1)
    return '\n'.join(lines[start:end])


In [49]:
# @title 7. Groq Client + Variant Generator (Cache + Per-Model Rate Limit)
from groq import Groq
from collections import deque
import time
import random

VARIANT_CACHE = OUTPUT_DIR / 'variant_cache.json'
if VARIANT_CACHE.exists():
    variant_cache = json.loads(VARIANT_CACHE.read_text(encoding='utf-8'))
else:
    variant_cache = {}

_GROQ_CLIENT = None
REQUEST_LOG_BY_MODEL = {}


def get_groq_client():
    global _GROQ_CLIENT
    if _GROQ_CLIENT is not None:
        return _GROQ_CLIENT

    try:
        api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        api_key = os.environ.get('GROQ_API_KEY')

    if not api_key:
        raise ValueError('GROQ_API_KEY not found in Colab Secrets or env.')

    _GROQ_CLIENT = Groq(api_key=api_key)
    return _GROQ_CLIENT


def get_model_rpm(model_name):
    return RPM_BY_MODEL.get(model_name, DEFAULT_RPM)


def throttle_requests(model_name):
    rpm = get_model_rpm(model_name)
    if rpm <= 0:
        return

    q = REQUEST_LOG_BY_MODEL.setdefault(model_name, deque())
    now = time.time()

    while q and now - q[0] > 60:
        q.popleft()

    if len(q) >= rpm:
        wait_s = 60 - (now - q[0]) + 0.1
        wait_s = max(wait_s, 0.1)
        print(f'Rate limit guard ({model_name}): sleeping {wait_s:.1f}s')
        time.sleep(wait_s)
        now = time.time()
        while q and now - q[0] > 60:
            q.popleft()

    q.append(time.time())


def safe_chat_completion(model_name, messages, expect_json=False, temperature=0.2):
    if not LLM_CALLS:
        return None

    client = get_groq_client()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            throttle_requests(model_name)
            kwargs = {
                'model': model_name,
                'messages': messages,
                'temperature': temperature
            }
            if expect_json:
                kwargs['response_format'] = {'type': 'json_object'}

            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if attempt >= MAX_RETRIES:
                print(f'LLM call failed ({model_name}) after {MAX_RETRIES} attempts: {e}')
                return None
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0.0, 0.5)
            print(f'LLM retry ({model_name}) {attempt}/{MAX_RETRIES}: {e}; sleeping {sleep_s:.1f}s')
            time.sleep(sleep_s)


def get_variants(concept):
    if concept in variant_cache:
        vals = variant_cache[concept]
        return vals[:MAX_VARIANTS_PER_CONCEPT]

    if not LLM_CALLS:
        vals = [concept]
        variant_cache[concept] = vals
        return vals

    system_prompt = (
        'You generate lexical variants and paraphrases for evidence retrieval. '
        'Return compact JSON: {"variants": ["..."]}.'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        'Return up to 12 variants including synonyms, abbreviations, paraphrases, and morphological forms. '
        'Keep each variant short.'
    )

    content = safe_chat_completion(
        model_name=MODEL_VARIANT_GEN,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.2
    )

    variants = [concept]
    if content:
        try:
            data = json.loads(content)
            llm_vars = data.get('variants', [])
            if isinstance(llm_vars, list):
                for item in llm_vars:
                    if isinstance(item, str) and item.strip():
                        variants.append(item.strip())
        except Exception:
            pass

    dedup = []
    seen = set()
    for v in variants:
        key = v.lower().strip()
        if not key or key in seen:
            continue
        seen.add(key)
        dedup.append(v)

    dedup = dedup[:MAX_VARIANTS_PER_CONCEPT]
    variant_cache[concept] = dedup
    VARIANT_CACHE.write_text(json.dumps(variant_cache, ensure_ascii=False, indent=2), encoding='utf-8')
    return dedup


In [50]:
# @title 8. Retrieval + Entailment Classification (Two-Model, Batched, Checkpointed)
def scan_lines_for_variants(lines, variants, fuzzy_threshold=85):
    hits = []
    for i, line in enumerate(lines):
        text = line.strip()
        if not text:
            continue
        low = text.lower()
        for v in variants:
            vlow = v.lower()
            if vlow in low:
                hits.append((i, line, v, 'lexical'))
                break
            score = fuzz.partial_ratio(vlow, low)
            if score >= fuzzy_threshold:
                hits.append((i, line, v, f'fuzzy:{score}'))
                break
    return hits


def clip_text(text, max_chars=MAX_CONTEXT_CHARS):
    if text is None:
        return ''
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + ' ...'


def chunk_list(items, n):
    for i in range(0, len(items), n):
        yield items[i:i+n]


def parse_batch_results(content, n):
    fallback = [{'label': 'WEAK', 'rationale': 'LLM parse failed'} for _ in range(n)]
    if not content:
        return fallback

    try:
        parsed = json.loads(content)
        results = parsed.get('results', [])
        mapped = {int(r['idx']): r for r in results if isinstance(r, dict) and 'idx' in r}
        out = []
        for i in range(n):
            r = mapped.get(i)
            if not r:
                out.append({'label': 'WEAK', 'rationale': 'No label'})
                continue
            label = str(r.get('label', 'WEAK')).upper().strip()
            if label not in {'DIRECT', 'INDIRECT', 'NONE'}:
                label = 'WEAK'
            out.append({'label': label, 'rationale': str(r.get('rationale', ''))})
        return out
    except Exception:
        return fallback


def classify_with_model(concept, contexts, model_name, hint_labels=None):
    compact = []
    for i, ctx in enumerate(contexts):
        row = {'idx': i, 'context': clip_text(ctx)}
        if hint_labels and i < len(hint_labels):
            row['hint_label'] = hint_labels[i]
        compact.append(row)

    system_prompt = (
        'You are an evidence auditor. '
        'For each snippet, decide if the concept is DIRECT, INDIRECT, or NONE. '
        'Return strict JSON object with key "results": '
        '[{"idx":0,"label":"DIRECT|INDIRECT|NONE","rationale":"..."}]'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        f'Snippets JSON:\n{json.dumps(compact, ensure_ascii=False)}'
    )

    content = safe_chat_completion(
        model_name=model_name,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.1
    )

    return parse_batch_results(content, len(contexts))


def classify_hits_batch(concept, contexts):
    if not contexts:
        return []

    if not LLM_CALLS:
        return [{
            'label': 'WEAK',
            'rationale': 'LLM disabled',
            'label_pass1': 'WEAK',
            'rationale_pass1': 'LLM disabled',
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        } for _ in contexts]

    pass1 = classify_with_model(concept, contexts, MODEL_PASS1)
    out = []
    for r in pass1:
        out.append({
            'label': r.get('label', 'WEAK'),
            'rationale': r.get('rationale', ''),
            'label_pass1': r.get('label', 'WEAK'),
            'rationale_pass1': r.get('rationale', ''),
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        })

    if USE_ESCALATION:
        idxs = [i for i, r in enumerate(out) if r['label'] in ESCALATE_LABELS]
        if idxs:
            sub_contexts = [contexts[i] for i in idxs]
            hints = [out[i]['label_pass1'] for i in idxs]
            pass2 = classify_with_model(concept, sub_contexts, MODEL_PASS2, hint_labels=hints)

            for j, i in enumerate(idxs):
                r2 = pass2[j]
                out[i]['label_pass2'] = r2.get('label', 'WEAK')
                out[i]['rationale_pass2'] = r2.get('rationale', '')
                out[i]['model_pass2'] = MODEL_PASS2
                out[i]['escalated'] = True

                if r2.get('label') in {'DIRECT', 'INDIRECT', 'NONE'}:
                    out[i]['label'] = r2.get('label')
                    out[i]['rationale'] = r2.get('rationale', '')

    return out


def classify_hits_chunked(concept, contexts):
    out = []
    for chunk in chunk_list(contexts, CLASSIFY_CHUNK_SIZE):
        out.extend(classify_hits_batch(concept, chunk))
    return out


def append_rows_csv(out_csv, rows):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    if out_csv.exists():
        df_new.to_csv(out_csv, mode='a', header=False, index=False)
    else:
        df_new.to_csv(out_csv, index=False)


def checkpoint_path(section_name):
    return CHECKPOINT_DIR / f'{section_name}_done_ids.json'


def load_done_ids(section_name):
    if not RESUME:
        return set()
    cp = checkpoint_path(section_name)
    if not cp.exists():
        return set()
    try:
        data = json.loads(cp.read_text(encoding='utf-8'))
        return set(data)
    except Exception:
        return set()


def save_done_ids(section_name, done_ids):
    cp = checkpoint_path(section_name)
    cp.write_text(json.dumps(sorted(list(done_ids)), ensure_ascii=False, indent=2), encoding='utf-8')


def process_in_batches(records):
    for batch in chunk_list(records, BATCH_SIZE_PAPERS):
        yield batch


def llm_fields_from_cls(cls):
    return {
        'llm_model_pass1': cls.get('model_pass1', ''),
        'llm_label_pass1': cls.get('label_pass1', ''),
        'llm_model_pass2': cls.get('model_pass2', ''),
        'llm_label_pass2': cls.get('label_pass2', ''),
        'llm_escalated': cls.get('escalated', False),
    }


In [51]:
# @title 9. Section 4A Evidence (Design Principles + Axes)
section_name = 'section4A'
out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'

done_ids = load_done_ids(section_name)
pending = [p for p in papers if p['paper_id'] not in done_ids]
print(f'{section_name}: pending papers = {len(pending)}')

axis_concepts = [
    'medium classification',
    'integration mechanism',
    'signal dimension',
    'detection type',
    'sensing task type',
    'measurement plane',
    'resolution versus accuracy'
]
axis_variants = {c: get_variants(c) for c in axis_concepts}

for batch in process_in_batches(pending):
    batch_rows = []
    for paper in tqdm(batch, desc=f'{section_name} batch'):
        paper_id = paper['paper_id']
        lines = paper['lines']
        heading_map = build_heading_map(lines)
        json_data = json_index.get(paper_id, {})

        for concept, variants in axis_variants.items():
            hits = scan_lines_for_variants(lines, variants)
            hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(concept, contexts)

            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': '4A',
                    'concept': concept,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': ''
                    , **llm_fields_from_cls(cls)
                })

        if isinstance(json_data, dict):
            clsf = json_data.get('study_level', {}).get('classification', {})
            if isinstance(clsf, dict):
                for k, v in clsf.items():
                    if v not in (None, '', 'NR'):
                        batch_rows.append({
                            'paper_id': paper_id,
                            'section': '4A',
                            'concept': f'json:{k}',
                            'variant': '',
                            'match_type': 'json',
                            'strength': 'DIRECT',
                            'rationale': 'Structured classification field',
                            'quote': '',
                            'line_start': '',
                            'line_end': '',
                            'heading_path': '',
                            'json_path': f'study_level.classification.{k}',
                            'json_value': str(v)
                            , 'llm_model_pass1': 'json'
                            , 'llm_label_pass1': 'DIRECT'
                            , 'llm_model_pass2': ''
                            , 'llm_label_pass2': ''
                            , 'llm_escalated': False
                        })

        done_ids.add(paper_id)

    append_rows_csv(out_csv, batch_rows)
    save_done_ids(section_name, done_ids)
    print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

print('Saved:', out_csv)


section4A: pending papers = 0
Saved: analysis/IV_ev_v2/section4A_evidence.csv


In [52]:
# @title 10. Section 4B Evidence (Medium-Based Classes)
section_name = 'section4B'
out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'

done_ids = load_done_ids(section_name)
pending = [p for p in papers if p['paper_id'] not in done_ids]
print(f'{section_name}: pending papers = {len(pending)}')

medium_concepts = {
    'fiber': ['fiber', 'fibre', 'das', 'otdr', 'ofdr'],
    'fso': ['free-space optical', 'fso', 'atmospheric', 'turbulence'],
    'vlc': ['visible light', 'vlc', 'lifi', 'led', 'im/dd'],
    'photonic_thz': ['photonic thz', 'optical-thz', 'thz', 'photomixing'],
    'hybrid': ['fiber-wireless', 'hybrid', 'mixed media', 'bridging']
}

medium_variants = {}
for k, terms in medium_concepts.items():
    expanded = []
    for t in terms:
        expanded.extend(get_variants(t))
    dedup = []
    seen = set()
    for e in expanded:
        key = e.lower().strip()
        if key and key not in seen:
            seen.add(key)
            dedup.append(e)
    medium_variants[k] = dedup[:MAX_VARIANTS_PER_CONCEPT]

for batch in process_in_batches(pending):
    batch_rows = []
    for paper in tqdm(batch, desc=f'{section_name} batch'):
        paper_id = paper['paper_id']
        lines = paper['lines']
        heading_map = build_heading_map(lines)
        json_data = json_index.get(paper_id, {})

        for medium, variants in medium_variants.items():
            hits = scan_lines_for_variants(lines, variants)
            hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(medium, contexts)

            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': '4B',
                    'concept': medium,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': ''
                    , **llm_fields_from_cls(cls)
                })

        if isinstance(json_data, dict):
            clsf = json_data.get('study_level', {}).get('classification', {})
            if isinstance(clsf, dict):
                v = clsf.get('oisac_medium_class')
                if v:
                    batch_rows.append({
                        'paper_id': paper_id,
                        'section': '4B',
                        'concept': 'json:oisac_medium_class',
                        'variant': '',
                        'match_type': 'json',
                        'strength': 'DIRECT',
                        'rationale': 'Structured classification field',
                        'quote': '',
                        'line_start': '',
                        'line_end': '',
                        'heading_path': '',
                        'json_path': 'study_level.classification.oisac_medium_class',
                        'json_value': str(v)
                        , 'llm_model_pass1': 'json'
                        , 'llm_label_pass1': 'DIRECT'
                        , 'llm_model_pass2': ''
                        , 'llm_label_pass2': ''
                        , 'llm_escalated': False
                    })

        done_ids.add(paper_id)

    append_rows_csv(out_csv, batch_rows)
    save_done_ids(section_name, done_ids)
    print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

print('Saved:', out_csv)


section4B: pending papers = 0
Saved: analysis/IV_ev_v2/section4B_evidence.csv


In [53]:
# @title 11. Section 4C Evidence (Integration Mechanisms)
section_name = 'section4C'
out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'

def get_first_scenario_obj(scenario_level):
    if isinstance(scenario_level, list):
        if scenario_level and isinstance(scenario_level[0], dict):
            return scenario_level[0]
        return {}
    if isinstance(scenario_level, dict):
        return scenario_level
    return {}

done_ids = load_done_ids(section_name)
pending = [p for p in papers if p['paper_id'] not in done_ids]
print(f'{section_name}: pending papers = {len(pending)}')

mechanism_concepts = {
    'shared_waveform': ['shared waveform', 'joint waveform', 'co-designed waveform'],
    'shared_hardware': ['shared hardware', 'common transceiver', 'shared front-end'],
    'shared_resources': ['resource partition', 'time sharing', 'frequency sharing', 'power sharing'],
    'shared_processing': ['joint processing', 'cooperative processing', 'shared signal processing']
}

mech_variants = {}
for k, terms in mechanism_concepts.items():
    expanded = []
    for t in terms:
        expanded.extend(get_variants(t))
    dedup = []
    seen = set()
    for e in expanded:
        key = e.lower().strip()
        if key and key not in seen:
            seen.add(key)
            dedup.append(e)
    mech_variants[k] = dedup[:MAX_VARIANTS_PER_CONCEPT]

for batch in process_in_batches(pending):
    batch_rows = []
    for paper in tqdm(batch, desc=f'{section_name} batch'):
        paper_id = paper['paper_id']
        lines = paper['lines']
        heading_map = build_heading_map(lines)
        json_data = json_index.get(paper_id, {})

        for mech, variants in mech_variants.items():
            hits = scan_lines_for_variants(lines, variants)
            hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(mech, contexts)

            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': '4C',
                    'concept': mech,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': ''
                    , **llm_fields_from_cls(cls)
                })

        if isinstance(json_data, dict):
            scenario_obj = get_first_scenario_obj(json_data.get('scenario_level', []))
            integ = scenario_obj.get('integration', {}) if isinstance(scenario_obj, dict) else {}
            if isinstance(integ, dict):
                for k, v in integ.items():
                    if v not in (None, '', 'NR', 'Not Reported'):
                        batch_rows.append({
                            'paper_id': paper_id,
                            'section': '4C',
                            'concept': f'json:{k}',
                            'variant': '',
                            'match_type': 'json',
                            'strength': 'DIRECT',
                            'rationale': 'Structured integration field',
                            'quote': '',
                            'line_start': '',
                            'line_end': '',
                            'heading_path': '',
                            'json_path': f'scenario_level.integration.{k}',
                            'json_value': str(v)
                            , 'llm_model_pass1': 'json'
                            , 'llm_label_pass1': 'DIRECT'
                            , 'llm_model_pass2': ''
                            , 'llm_label_pass2': ''
                            , 'llm_escalated': False
                        })

        done_ids.add(paper_id)

    append_rows_csv(out_csv, batch_rows)
    save_done_ids(section_name, done_ids)
    print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

print('Saved:', out_csv)


section4C: pending papers = 0
Saved: analysis/IV_ev_v2/section4C_evidence.csv


In [54]:
# @title 12. Section 4D Evidence (Signal Dimension + Detection)
section_name = 'section4D'
out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'

def get_first_scenario_obj(scenario_level):
    if isinstance(scenario_level, list):
        if scenario_level and isinstance(scenario_level[0], dict):
            return scenario_level[0]
        return {}
    if isinstance(scenario_level, dict):
        return scenario_level
    return {}

done_ids = load_done_ids(section_name)
pending = [p for p in papers if p['paper_id'] not in done_ids]
print(f'{section_name}: pending papers = {len(pending)}')

detect_concepts = {
    'imdd': ['im/dd', 'intensity modulation', 'direct detection'],
    'coherent': ['coherent detection', 'heterodyne', 'homodyne'],
    'intensity_only': ['intensity-only', 'nonnegative signal'],
    'complex_field': ['complex field', 'i/q', 'in-phase quadrature']
}

detect_variants = {}
for k, terms in detect_concepts.items():
    expanded = []
    for t in terms:
        expanded.extend(get_variants(t))
    dedup = []
    seen = set()
    for e in expanded:
        key = e.lower().strip()
        if key and key not in seen:
            seen.add(key)
            dedup.append(e)
    detect_variants[k] = dedup[:MAX_VARIANTS_PER_CONCEPT]

for batch in process_in_batches(pending):
    batch_rows = []
    for paper in tqdm(batch, desc=f'{section_name} batch'):
        paper_id = paper['paper_id']
        lines = paper['lines']
        heading_map = build_heading_map(lines)
        json_data = json_index.get(paper_id, {})

        for dkey, variants in detect_variants.items():
            hits = scan_lines_for_variants(lines, variants)
            hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(dkey, contexts)

            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': '4D',
                    'concept': dkey,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': ''
                    , **llm_fields_from_cls(cls)
                })

        if isinstance(json_data, dict):
            scenario_obj = get_first_scenario_obj(json_data.get('scenario_level', []))
            rx = scenario_obj.get('receiver', {}) if isinstance(scenario_obj, dict) else {}
            if isinstance(rx, dict):
                v = rx.get('rx_detection_type')
                if v not in (None, '', 'NR', 'Not Reported'):
                    batch_rows.append({
                        'paper_id': paper_id,
                        'section': '4D',
                        'concept': 'json:rx_detection_type',
                        'variant': '',
                        'match_type': 'json',
                        'strength': 'DIRECT',
                        'rationale': 'Structured receiver field',
                        'quote': '',
                        'line_start': '',
                        'line_end': '',
                        'heading_path': '',
                        'json_path': 'scenario_level.receiver.rx_detection_type',
                        'json_value': str(v)
                        , 'llm_model_pass1': 'json'
                        , 'llm_label_pass1': 'DIRECT'
                        , 'llm_model_pass2': ''
                        , 'llm_label_pass2': ''
                        , 'llm_escalated': False
                    })

        done_ids.add(paper_id)

    append_rows_csv(out_csv, batch_rows)
    save_done_ids(section_name, done_ids)
    print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

print('Saved:', out_csv)


section4D: pending papers = 0
Saved: analysis/IV_ev_v2/section4D_evidence.csv


In [55]:
# @title 13. Section 4E Summary Outputs (Taxonomy Skeletons)
# Build a simple hierarchy: medium -> mechanism -> detection
import collections
import re

def safe_get(d, path, default=None):
    cur = d
    for p in path.split('.'):
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return default
    return cur

def get_first_scenario(scenario_level):
    if isinstance(scenario_level, list):
        if scenario_level and isinstance(scenario_level[0], dict):
            return scenario_level[0]
        return {}
    if isinstance(scenario_level, dict):
        return scenario_level
    return {}

def _norm_token(value):
    s = str(value or '').strip()
    if not s:
        return 'unknown'
    low = s.lower()
    if low in {'nr', 'not reported', 'none', 'nan'}:
        return 'unknown'
    low = low.replace('/', '_')
    low = re.sub(r'[^a-z0-9_\-\s\[\],]+', '', low)
    low = re.sub(r'[\s\-]+', '_', low)
    low = re.sub(r'_+', '_', low).strip('_')
    return low or 'unknown'

def normalize_medium(value):
    s = _norm_token(value)
    alias = {
        'visible_light': 'wireless_vlc',
        'vlc': 'wireless_vlc',
        'rf': 'wireless_rf',
    }
    return alias.get(s, s)

def normalize_mechanism(value):
    s = _norm_token(value)
    alias = {
        'sharedfrontend': 'shared_frontend',
        'shared_front_end': 'shared_frontend',
        'shared_frontend': 'shared_frontend',
    }
    return alias.get(s, s)

def normalize_detection(value):
    s = _norm_token(value)
    alias = {
        'direct_detection': 'direct',
        'directdetection': 'direct',
        'direct': 'direct',
        'coherent_detection': 'coherent',
    }
    return alias.get(s, s)

def normalize_task(value):
    s = _norm_token(value)
    if s.startswith('[') and s.endswith(']'):
        s = s[1:-1].strip()
    s = s.replace("'", '').replace('"', '')
    s = s.replace(', ', '|').replace(',', '|')
    return s or 'unknown'

tree = collections.defaultdict(lambda: collections.defaultdict(lambda: collections.defaultdict(int)))
rows = []
for paper_id, j in json_index.items():
    clsf = safe_get(j, 'study_level.classification', {}) or {}
    scenarios = safe_get(j, 'scenario_level', [])
    sc0 = get_first_scenario(scenarios)

    medium_raw = clsf.get('oisac_medium_class', 'unknown')
    mech_raw = safe_get(sc0, 'integration.hardware_sharing_mode', 'unknown')
    rx_raw = safe_get(sc0, 'receiver.rx_detection_type', 'unknown')
    task_raw = safe_get(sc0, 'sensing_metrics.sensing_task_type', 'unknown')

    medium = normalize_medium(medium_raw) if NORMALIZE_LABELS else medium_raw
    mech = normalize_mechanism(mech_raw) if NORMALIZE_LABELS else mech_raw
    rx = normalize_detection(rx_raw) if NORMALIZE_LABELS else rx_raw
    task = normalize_task(task_raw) if NORMALIZE_LABELS else task_raw

    tree[medium][mech][rx] += 1
    rows.append({
        'paper_id': paper_id,
        'medium': medium,
        'mechanism': mech,
        'detection': rx,
        'sensing_task': task,
        'medium_raw': str(medium_raw),
        'mechanism_raw': str(mech_raw),
        'detection_raw': str(rx_raw),
        'sensing_task_raw': str(task_raw),
    })

# Save flat summary table
df = pd.DataFrame(rows).sort_values(by=['paper_id'])
table_path = OUTPUT_DIR / 'section4E_summary_table.csv'
df.to_csv(table_path, index=False)

# Save hierarchy as JSON
tree_path = OUTPUT_DIR / 'section4E_taxonomy_tree.json'
tree_path.write_text(json.dumps(tree, indent=2), encoding='utf-8')

print('Saved:', table_path)
print('Saved:', tree_path)


Saved: analysis/IV_ev_v2/section4E_summary_table.csv
Saved: analysis/IV_ev_v2/section4E_taxonomy_tree.json


In [56]:
# @title 14. Post-Processing Artifacts (Graph + Anchors + Clusters + Contract Audit)
import hashlib
import re
from collections import defaultdict

def _norm_token(value):
    s = str(value or '').strip()
    if not s:
        return 'unknown'
    low = s.lower()
    if low in {'nr', 'not reported', 'none', 'nan'}:
        return 'unknown'
    low = low.replace('/', '_')
    low = re.sub(r'[^a-z0-9_\-\s\[\],]+', '', low)
    low = re.sub(r'[\s\-]+', '_', low)
    low = re.sub(r'_+', '_', low).strip('_')
    return low or 'unknown'

def normalize_medium(value):
    s = _norm_token(value)
    alias = {'visible_light': 'wireless_vlc', 'vlc': 'wireless_vlc', 'rf': 'wireless_rf'}
    return alias.get(s, s)

def normalize_mechanism(value):
    s = _norm_token(value)
    alias = {'sharedfrontend': 'shared_frontend', 'shared_front_end': 'shared_frontend'}
    return alias.get(s, s)

def normalize_detection(value):
    s = _norm_token(value)
    alias = {'direct_detection': 'direct', 'directdetection': 'direct', 'coherent_detection': 'coherent'}
    return alias.get(s, s)

def normalize_task(value):
    s = _norm_token(value)
    if s.startswith('[') and s.endswith(']'):
        s = s[1:-1].strip()
    s = s.replace("'", '').replace('"', '')
    s = s.replace(', ', '|').replace(',', '|')
    return s or 'unknown'

def get_first_scenario_obj(scenario_level):
    if isinstance(scenario_level, list):
        if scenario_level and isinstance(scenario_level[0], dict):
            return scenario_level[0]
        return {}
    if isinstance(scenario_level, dict):
        return scenario_level
    return {}

def as_int_or_blank(x):
    try:
        if pd.isna(x) or x == '':
            return ''
        return int(float(x))
    except Exception:
        return ''

sec_frames = {}
for sec in ['A', 'B', 'C', 'D']:
    p = OUTPUT_DIR / f'section4{sec}_evidence.csv'
    sec_frames[sec] = pd.read_csv(p) if p.exists() else pd.DataFrame()

# 1) retrieval_hits.jsonl (lexical/fuzzy rows)
retrieval_path = OUTPUT_DIR / 'retrieval_hits.jsonl'
with retrieval_path.open('w', encoding='utf-8') as f:
    for sec, df in sec_frames.items():
        if df.empty:
            continue
        for _, r in df.iterrows():
            mt = str(r.get('match_type', ''))
            if mt.lower() == 'json':
                continue
            rec = {
                'paper_id': str(r.get('paper_id', '')),
                'section': f'4{sec}',
                'concept': str(r.get('concept', '')),
                'variant': str(r.get('variant', '')),
                'match_type': mt,
                'quote': str(r.get('quote', '')),
                'line_start': as_int_or_blank(r.get('line_start', '')),
                'line_end': as_int_or_blank(r.get('line_end', '')),
                'heading_path': str(r.get('heading_path', '')),
                'strength': str(r.get('strength', '')),
                'rationale': str(r.get('rationale', '')),
                'llm_model_pass1': str(r.get('llm_model_pass1', '')),
                'llm_label_pass1': str(r.get('llm_label_pass1', '')),
                'llm_model_pass2': str(r.get('llm_model_pass2', '')),
                'llm_label_pass2': str(r.get('llm_label_pass2', '')),
                'llm_escalated': bool(r.get('llm_escalated', False)),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Saved:', retrieval_path)

# 2) anchor_table.csv
anchor_rows = []
for sec, df in sec_frames.items():
    if df.empty:
        continue
    for _, r in df.iterrows():
        concept = str(r.get('concept', '')).strip()
        claim_key = f'4{sec}|{concept}'
        claim_id = hashlib.sha1(claim_key.encode('utf-8')).hexdigest()[:12]
        anchor_rows.append({
            'claim_id': claim_id,
            'claim_key': claim_key,
            'section': f'4{sec}',
            'paper_id': str(r.get('paper_id', '')),
            'concept': concept,
            'variant': str(r.get('variant', '')),
            'match_type': str(r.get('match_type', '')),
            'strength': str(r.get('strength', '')),
            'rationale': str(r.get('rationale', '')),
            'quote': str(r.get('quote', '')),
            'line_start': as_int_or_blank(r.get('line_start', '')),
            'line_end': as_int_or_blank(r.get('line_end', '')),
            'heading_path': str(r.get('heading_path', '')),
            'json_path': str(r.get('json_path', '')),
            'json_value': str(r.get('json_value', '')),
        })

anchor_cols = [
    'claim_id', 'claim_key', 'section', 'paper_id', 'concept', 'variant', 'match_type',
    'strength', 'rationale', 'quote', 'line_start', 'line_end', 'heading_path',
    'json_path', 'json_value', 'claim_supported'
]
if anchor_rows:
    anchor_df = pd.DataFrame(anchor_rows)
    agg = anchor_df.assign(
        is_direct=anchor_df['strength'].eq('DIRECT'),
        is_indirect=anchor_df['strength'].eq('INDIRECT')
    ).groupby('claim_id', as_index=False)[['is_direct', 'is_indirect']].sum()
    agg['claim_supported'] = (agg['is_direct'] >= 1) | (agg['is_indirect'] >= 2)
    anchor_df = anchor_df.merge(agg[['claim_id', 'claim_supported']], on='claim_id', how='left')
    anchor_df = anchor_df[anchor_cols]
else:
    anchor_df = pd.DataFrame(columns=anchor_cols)

anchor_path = OUTPUT_DIR / 'anchor_table.csv'
anchor_df.to_csv(anchor_path, index=False)
print('Saved:', anchor_path)

# 3) evidence_graph.jsonl
anchors_by_paper = defaultdict(list)
for _, r in anchor_df.iterrows():
    anchors_by_paper[str(r.get('paper_id', ''))].append({
        'claim_id': str(r.get('claim_id', '')),
        'section': str(r.get('section', '')),
        'concept': str(r.get('concept', '')),
        'strength': str(r.get('strength', '')),
        'heading_path': str(r.get('heading_path', '')),
        'line_start': as_int_or_blank(r.get('line_start', '')),
        'line_end': as_int_or_blank(r.get('line_end', '')),
        'json_path': str(r.get('json_path', '')),
    })

graph_path = OUTPUT_DIR / 'evidence_graph.jsonl'
with graph_path.open('w', encoding='utf-8') as f:
    for paper_id, j in sorted(json_index.items()):
        clsf = ((j or {}).get('study_level') or {}).get('classification', {}) if isinstance(j, dict) else {}
        sc0 = get_first_scenario_obj((j or {}).get('scenario_level', [])) if isinstance(j, dict) else {}
        integ = sc0.get('integration', {}) if isinstance(sc0, dict) else {}
        recv = sc0.get('receiver', {}) if isinstance(sc0, dict) else {}
        sens = sc0.get('sensing_metrics', {}) if isinstance(sc0, dict) else {}

        medium_raw = clsf.get('oisac_medium_class', 'unknown') if isinstance(clsf, dict) else 'unknown'
        mechanism_raw = integ.get('hardware_sharing_mode', 'unknown') if isinstance(integ, dict) else 'unknown'
        detection_raw = recv.get('rx_detection_type', 'unknown') if isinstance(recv, dict) else 'unknown'
        task_raw = sens.get('sensing_task_type', 'unknown') if isinstance(sens, dict) else 'unknown'

        rec = {
            'paper_id': paper_id,
            'structured': {
                'medium_raw': str(medium_raw),
                'medium': normalize_medium(medium_raw) if NORMALIZE_LABELS else str(medium_raw),
                'mechanism_raw': str(mechanism_raw),
                'mechanism': normalize_mechanism(mechanism_raw) if NORMALIZE_LABELS else str(mechanism_raw),
                'detection_raw': str(detection_raw),
                'detection': normalize_detection(detection_raw) if NORMALIZE_LABELS else str(detection_raw),
                'sensing_task_raw': str(task_raw),
                'sensing_task': normalize_task(task_raw) if NORMALIZE_LABELS else str(task_raw),
            },
            'anchor_count': len(anchors_by_paper.get(paper_id, [])),
            'anchors': anchors_by_paper.get(paper_id, []),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Saved:', graph_path)

# 4) axis_definitions.md + mapping_rules.md
observed_medium = sorted({normalize_medium(((j.get('study_level') or {}).get('classification') or {}).get('oisac_medium_class', 'unknown')) for j in json_index.values() if isinstance(j, dict)})
observed_mech = sorted({normalize_mechanism((get_first_scenario_obj((j or {}).get('scenario_level', [])).get('integration', {}) or {}).get('hardware_sharing_mode', 'unknown')) for j in json_index.values() if isinstance(j, dict)})
observed_det = sorted({normalize_detection((get_first_scenario_obj((j or {}).get('scenario_level', [])).get('receiver', {}) or {}).get('rx_detection_type', 'unknown')) for j in json_index.values() if isinstance(j, dict)})
observed_task = sorted({normalize_task((get_first_scenario_obj((j or {}).get('scenario_level', [])).get('sensing_metrics', {}) or {}).get('sensing_task_type', 'unknown')) for j in json_index.values() if isinstance(j, dict)})

axis_md = []
axis_md.append('# Section 4 Axis Definitions (v2)')
axis_md.append('')
axis_md.append('This file freezes taxonomy axes using Section II schema/metric contract and Section IV evidence extraction.')
axis_md.append('')
axis_md.append('## Axis 1: Medium')
axis_md.append('Normalized labels: ' + ', '.join(observed_medium))
axis_md.append('')
axis_md.append('## Axis 2: Integration Mechanism')
axis_md.append('Normalized labels: ' + ', '.join(observed_mech))
axis_md.append('')
axis_md.append('## Axis 3: Detection / Signal Plane')
axis_md.append('Normalized labels: ' + ', '.join(observed_det))
axis_md.append('')
axis_md.append('## Axis 4: Sensing Task Class')
axis_md.append('Normalized labels: ' + ', '.join(observed_task))
axis_md.append('')
axis_md.append('## Contract Notes')
axis_md.append('- Keep optical-plane and electrical-plane metrics separated.')
axis_md.append('- Keep resolution and accuracy as separate fields.')

axis_path = OUTPUT_DIR / 'axis_definitions.md'
axis_path.write_text('\n'.join(axis_md), encoding='utf-8')
print('Saved:', axis_path)

map_md = []
map_md.append('# Section 4 Mapping Rules (v2)')
map_md.append('')
map_md.append('1. Primary axis assignment uses structured JSON field when available.')
map_md.append('2. If JSON missing, use strongest anchor evidence (DIRECT > INDIRECT > NONE).')
map_md.append('3. For multi-task labels, split by `|` and retain all; primary is first token.')
map_md.append('4. Hybrid media remain `hybrid`; no forced decomposition.')
map_md.append('5. Label normalization applies before clustering (`NORMALIZE_LABELS=True`).')
map_md.append('6. Contradictory anchors are kept and flagged via `contract_violations.csv`.')

map_path = OUTPUT_DIR / 'mapping_rules.md'
map_path.write_text('\n'.join(map_md), encoding='utf-8')
print('Saved:', map_path)

# 5) cluster_map.csv
cluster_rows = []
for paper_id, j in sorted(json_index.items()):
    clsf = ((j or {}).get('study_level') or {}).get('classification', {}) if isinstance(j, dict) else {}
    sc0 = get_first_scenario_obj((j or {}).get('scenario_level', [])) if isinstance(j, dict) else {}
    integ = sc0.get('integration', {}) if isinstance(sc0, dict) else {}
    recv = sc0.get('receiver', {}) if isinstance(sc0, dict) else {}
    sens = sc0.get('sensing_metrics', {}) if isinstance(sc0, dict) else {}

    medium = normalize_medium((clsf or {}).get('oisac_medium_class', 'unknown'))
    mech = normalize_mechanism((integ or {}).get('hardware_sharing_mode', 'unknown'))
    det = normalize_detection((recv or {}).get('rx_detection_type', 'unknown'))
    task = normalize_task((sens or {}).get('sensing_task_type', 'unknown'))

    paper_anchors = anchors_by_paper.get(paper_id, [])
    evidence_refs = sorted({f"{a.get('section', '')}:{a.get('concept', '')}" for a in paper_anchors if a.get('section') and a.get('concept')})
    anchor_count = len(paper_anchors)
    if anchor_count >= 8:
        conf = 'high'
    elif anchor_count >= 3:
        conf = 'medium'
    else:
        conf = 'low'

    cluster_rows.append({
        'paper_id': paper_id,
        'medium': medium,
        'mechanism': mech,
        'detection': det,
        'sensing_task': task,
        'anchor_count': anchor_count,
        'confidence': conf,
        'evidence_refs': '; '.join(evidence_refs[:20]),
    })

cluster_df = pd.DataFrame(cluster_rows)
cluster_path = OUTPUT_DIR / 'cluster_map.csv'
cluster_df.to_csv(cluster_path, index=False)
print('Saved:', cluster_path)

# 6) contract_violations.csv
violations = []
if not anchor_df.empty:
    for _, r in anchor_df.iterrows():
        quote = str(r.get('quote', ''))
        if not quote or quote.lower() == 'nan':
            continue
        q = quote.lower()
        paper_id = str(r.get('paper_id', ''))

        has_osnr = 'osnr' in q
        has_snr = bool(re.search(r'\besnr\b|\bsnr\b', q))
        has_model_hint = any(k in q for k in ['receiver model', 'noise model', 'post-detection', 'conversion model', 'photodetector'])
        if has_osnr and has_snr and not has_model_hint:
            violations.append({
                'paper_id': paper_id,
                'section': str(r.get('section', '')),
                'concept': str(r.get('concept', '')),
                'category': 'METRIC_PLANE',
                'severity': 'MAJOR',
                'reason': 'OSNR and SNR/ESNR mentioned together without explicit conversion/model context.',
                'quote': quote,
                'line_start': as_int_or_blank(r.get('line_start', '')),
                'heading_path': str(r.get('heading_path', '')),
            })

        has_resolution = any(k in q for k in ['resolution', 'delta_r', 'delta z', 'range_resolution'])
        has_accuracy = any(k in q for k in ['accuracy', 'sigma_r', 'range_accuracy'])
        if has_resolution and has_accuracy and 'respectively' not in q:
            violations.append({
                'paper_id': paper_id,
                'section': str(r.get('section', '')),
                'concept': str(r.get('concept', '')),
                'category': 'METRIC_ALIASING',
                'severity': 'MAJOR',
                'reason': 'Resolution and accuracy co-occur without explicit separation context.',
                'quote': quote,
                'line_start': as_int_or_blank(r.get('line_start', '')),
                'heading_path': str(r.get('heading_path', '')),
            })

viol_cols = ['paper_id', 'section', 'concept', 'category', 'severity', 'reason', 'quote', 'line_start', 'heading_path']
viol_df = pd.DataFrame(violations, columns=viol_cols)
viol_path = OUTPUT_DIR / 'contract_violations.csv'
viol_df.to_csv(viol_path, index=False)
print('Saved:', viol_path)
print('Violations:', len(viol_df))


Saved: analysis/IV_ev_v2/retrieval_hits.jsonl
Saved: analysis/IV_ev_v2/anchor_table.csv
Saved: analysis/IV_ev_v2/evidence_graph.jsonl
Saved: analysis/IV_ev_v2/axis_definitions.md
Saved: analysis/IV_ev_v2/mapping_rules.md
Saved: analysis/IV_ev_v2/cluster_map.csv
Saved: analysis/IV_ev_v2/contract_violations.csv
Violations: 84


In [57]:
# @title 15. Readiness Report
report = []
for fname in [
    'section4A_evidence.csv',
    'section4B_evidence.csv',
    'section4C_evidence.csv',
    'section4D_evidence.csv',
    'section4E_summary_table.csv',
    'section4E_taxonomy_tree.json',
    'evidence_graph.jsonl',
    'retrieval_hits.jsonl',
    'anchor_table.csv',
    'axis_definitions.md',
    'mapping_rules.md',
    'cluster_map.csv',
    'contract_violations.csv',
]:
    p = OUTPUT_DIR / fname
    report.append(f'{fname}: ' + ('OK' if p.exists() else 'MISSING'))

report_path = OUTPUT_DIR / 'readiness_report.md'
report_path.write_text('\n'.join(report), encoding='utf-8')
print('\n'.join(report))
print('Saved:', report_path)


section4A_evidence.csv: OK
section4B_evidence.csv: OK
section4C_evidence.csv: OK
section4D_evidence.csv: OK
section4E_summary_table.csv: OK
section4E_taxonomy_tree.json: OK
evidence_graph.jsonl: OK
retrieval_hits.jsonl: OK
anchor_table.csv: OK
axis_definitions.md: OK
mapping_rules.md: OK
cluster_map.csv: OK
contract_violations.csv: OK
Saved: analysis/IV_ev_v2/readiness_report.md
